# SpliceAImod on Colab — sharded, resumable A100 run

Runs the PyTorch SpliceAI fork over a large BCF on a single Colab GPU, in shards, with
finished shards persisted to Google Drive so a runtime disconnect only costs the shard in flight.

**Runtime:** `Runtime → Change runtime type → A100 GPU`, High-RAM.

Design notes:

- Everything hot lives on `/content` (local SSD): the reference FASTA, the shards, the tmpdir,
  and the in-progress output. Only *finished* TSVs go to Drive. pyfaidx random access through the
  Drive FUSE mount is far too slow, and streaming a gzip writer to Drive can leave truncated files.
- `--precision fp16` is passed explicitly. `auto` resolves to bf16 on Ampere+, which benchmarks slower
  for this conv-only model.
- `-G all` is used, not `-G 0`: in this codebase `-G 0` means *CPU mode* (see `initialize_devices`), not "GPU index 0".
- The driver runs as a detached `nohup` process with its log on Drive, so it survives cell interrupts
  and short browser disconnects. Re-running the launch cell after a disconnect resumes at the first
  shard without a finished file on Drive.

Cells 1–5 are per-session setup (~5 min with a cached reference). Cell 7 launches, cell 8 monitors.


## 1. Config

In [ ]:
import os, json

CFG = dict(
    # ---- Paths (persistent, on Drive) ----------------------------------------
    DRIVE_ROOT   = "/content/drive/MyDrive/spliceai_run",
    INPUT_BCF    = "/content/drive/MyDrive/spliceai_run/input/variants.bcf",  # .bcf or .vcf.gz, index alongside
    REF_ON_DRIVE = "/content/drive/MyDrive/spliceai_run/ref/GRCh38.fa",       # cached FASTA + .fai; downloaded if missing
    REF_URL      = ("https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/000/001/405/GCA_000001405.15_GRCh38/"
                    "seqs_for_alignment_pipelines.ucsc_ids/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna.gz"),
    REPO_URL     = "https://github.com/chundruv/SpliceAImod.git",
    ANNOTATION   = "gencodev49",     # or "MANEv1.4"
    DISTANCE     = 500,

    # ---- Sharding ----------------------------------------------------------
    # Variants per shard. Predictions per variant ~ alleles x overlapping transcripts, so expect
    # ~2-3x this many predictions. Aim for shards that finish in 2-3 h; at ~2M pred/h (A100, fp16)
    # that is on the order of 1-2M variants. Start smaller for the first run and adjust.
    VARIANTS_PER_SHARD = 1_500_000,

    # ---- Inference ---------------------------------------------------------
    PRED_BATCH    = 20480,   # -B
    TORCH_BATCH   = 512,     # -T : fused 5-model ensemble => ~5x activation memory per sample.
                             #      512 fits A100 40GB; try 768-1024 on 40GB, 1536-2048 on 80GB
    BATCH_WORKERS = 4,       # --batch-workers ; A100 runtime has ~12 vCPUs
    PRECISION     = "fp16",
    EXTRA_FLAGS   = "--compile",   # add --no-cuda-graphs only if graph capture OOMs

    # ---- Local (ephemeral) layout --------------------------------------------
    LOCAL        = "/content/work",
)
CFG.update(
    LOCAL_REF    = f"{CFG['LOCAL']}/ref/GRCh38.fa",
    LOCAL_SHARDS = f"{CFG['LOCAL']}/shards",
    LOCAL_OUT    = f"{CFG['LOCAL']}/out",
    LOCAL_TMP    = f"{CFG['LOCAL']}/tmp",
    DRIVE_OUT    = f"{CFG['DRIVE_ROOT']}/out",
    DRIVE_LOGS   = f"{CFG['DRIVE_ROOT']}/logs",
    MANIFEST     = f"{CFG['DRIVE_ROOT']}/shards.json",
)
for d in (f"{CFG['LOCAL']}/ref", CFG["LOCAL_SHARDS"], CFG["LOCAL_OUT"], CFG["LOCAL_TMP"]):
    os.makedirs(d, exist_ok=True)
json.dump(CFG, open("/content/run_config.json", "w"), indent=1)
globals().update(CFG)
print("config written to /content/run_config.json")


## 2. Mount Drive, check the GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for d in (DRIVE_OUT, DRIVE_LOGS, f"{DRIVE_ROOT}/input", f"{DRIVE_ROOT}/ref"):
    os.makedirs(d, exist_ok=True)

import torch, subprocess
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU — change the runtime type to A100"
name = torch.cuda.get_device_name(0)
if "A100" not in name:
    print(f"WARNING: running on {name}, not an A100 — TORCH_BATCH is tuned for A100")


## 3. Install

Colab ships its own torch; it is kept unless it is too old for `torch.compile` (a reinstall costs
10+ min and several GB every session). `bcftools`/`samtools` come from apt for sharding and faidx.

In [ ]:
%%bash
set -e
REPO_URL=$(python -c "import json;print(json.load(open('/content/run_config.json'))['REPO_URL'])")
if ! python - <<'PY'
import torch, sys
maj, mnr = map(int, torch.__version__.split("+")[0].split(".")[:2])
sys.exit(0 if (maj, mnr) >= (2, 4) else 1)
PY
then echo "torch too old for torch.compile — reinstalling"; pip install -q --upgrade torch --index-url https://download.pytorch.org/whl/cu128; fi
python -c "import torch; print('torch', torch.__version__)"

apt-get -qq update && apt-get -qq install -y bcftools samtools pigz > /dev/null
pip install -q pysam pyfaidx pandas numpy intervaltree numba h5py psutil nvidia-ml-py

cd /content
[ -d SpliceAImod ] || git clone -q "$REPO_URL"
cd SpliceAImod && git checkout bench && pip install -q -e .
which spliceai


## 4. Reference genome → local disk

Uses the Drive copy if present, otherwise downloads from NCBI once and caches it on Drive.
pyfaidx needs an uncompressed (or bgzipped) FASTA with a `.fai`.

In [ ]:
%%bash
set -e
eval "$(python -c "import json;c=json.load(open('/content/run_config.json'));print(f'REF_ON_DRIVE={c[\"REF_ON_DRIVE\"]};LOCAL_REF={c[\"LOCAL_REF\"]};REF_URL={c[\"REF_URL\"]}')")"
if [ -f "$LOCAL_REF.fai" ]; then echo "local ref already present"; exit 0; fi
if [ -f "$REF_ON_DRIVE" ] && [ -f "$REF_ON_DRIVE.fai" ]; then
  echo "copying ref from Drive"; cp "$REF_ON_DRIVE" "$LOCAL_REF"; cp "$REF_ON_DRIVE.fai" "$LOCAL_REF.fai"
else
  echo "downloading ref from NCBI"; wget -q -O - "$REF_URL" | pigz -dc > "$LOCAL_REF"
  samtools faidx "$LOCAL_REF"
  mkdir -p "$(dirname "$REF_ON_DRIVE")"
  cp "$LOCAL_REF" "$REF_ON_DRIVE"; cp "$LOCAL_REF.fai" "$REF_ON_DRIVE.fai"
  echo "cached ref on Drive"
fi
ls -lh "$LOCAL_REF"*


## 5. Shard the input

Splits by *variant count*, cutting only between distinct positions, so multi-allelic records at one
position never straddle a shard. The shard manifest (regions) is saved to Drive so the split is
deterministic across sessions; the shard BCFs are rebuilt on `/content` each session — a cheap
region query on the indexed input.

In [ ]:
import subprocess, shutil

local_input = f"{LOCAL}/input{os.path.splitext(INPUT_BCF)[1]}"
if not os.path.exists(local_input):
    shutil.copy(INPUT_BCF, local_input)
    for ext in (".csi", ".tbi"):
        if os.path.exists(INPUT_BCF + ext):
            shutil.copy(INPUT_BCF + ext, local_input + ext)
if not (os.path.exists(local_input + ".csi") or os.path.exists(local_input + ".tbi")):
    subprocess.run(["bcftools", "index", local_input], check=True)

if os.path.exists(MANIFEST):
    shards = json.load(open(MANIFEST))
    print(f"loaded existing manifest: {len(shards)} shards")
else:
    print("scanning positions …")
    proc = subprocess.Popen(["bcftools", "query", "-f", "%CHROM\t%POS\n", local_input],
                            stdout=subprocess.PIPE, text=True)
    raw, cur_chrom, start, count, last_pos = [], None, None, 0, None
    for line in proc.stdout:
        chrom, pos = line.rstrip("\n").split("\t"); pos = int(pos)
        if chrom != cur_chrom:
            if cur_chrom is not None:
                raw.append((cur_chrom, start, last_pos))
            cur_chrom, start, count = chrom, pos, 0
        elif count >= VARIANTS_PER_SHARD and pos != last_pos:
            raw.append((cur_chrom, start, last_pos))
            start, count = pos, 0
        count += 1; last_pos = pos
    if cur_chrom is not None:
        raw.append((cur_chrom, start, last_pos))
    proc.wait()
    shards = [{"id": f"shard_{i:04d}", "chrom": c, "start": s, "end": e} for i, (c, s, e) in enumerate(raw)]
    json.dump(shards, open(MANIFEST, "w"), indent=1)
    print(f"wrote manifest: {len(shards)} shards")

todo = [s for s in shards if not os.path.exists(f"{DRIVE_OUT}/{s['id']}.tsv.gz")]
print(f"{len(todo)} shards still to run")
for s in todo:
    bcf = f"{LOCAL_SHARDS}/{s['id']}.bcf"
    if os.path.exists(bcf + ".csi"):
        continue
    subprocess.run(["bcftools", "view", "-r", f"{s['chrom']}:{s['start']}-{s['end']}",
                    "-Ob", "-o", bcf, local_input], check=True)
    subprocess.run(["bcftools", "index", bcf], check=True)
print("shards ready")


## 6. Smoke test (optional, ~1 min)

Runs the bundled example so compile / graph-capture problems surface here, not two hours into shard 0.

In [ ]:
!cd /content/SpliceAImod && spliceai -I examples/input.vcf -O /content/smoke.tsv.gz \
  -R {LOCAL_REF} -A {ANNOTATION} -D {DISTANCE} -G all --precision {PRECISION} {EXTRA_FLAGS} \
  -B 1024 -T 256 --batch-workers 1 -t {LOCAL_TMP} -V 2>&1 | tail -20
!zcat /content/smoke.tsv.gz | head -5 | cut -f1-10


## 7. Driver (detached, resumable)

Each shard: run → write to `/content` → move to Drive as `.partial` → rename to final.
The rename is the commit; a `.partial` on Drive is *not* treated as done, so a disconnect
mid-copy is safe. A failed shard stops the driver so the failure is visible in the log.

In [ ]:
%%writefile /content/colab_driver.py
import os, sys, json, shutil, subprocess, time, glob
C = json.load(open("/content/run_config.json"))

def log(msg): print(time.strftime("%Y-%m-%d %H:%M:%S"), msg, flush=True)

shards = json.load(open(C["MANIFEST"]))
for s in shards:
    sid = s["id"]
    final = f"{C['DRIVE_OUT']}/{sid}.tsv.gz"
    if os.path.exists(final):
        continue
    bcf = f"{C['LOCAL_SHARDS']}/{sid}.bcf"
    if not os.path.exists(bcf):
        log(f"{sid}: shard BCF missing on local disk — re-run the sharding cell"); sys.exit(2)
    out = f"{C['LOCAL_OUT']}/{sid}.tsv.gz"
    for stale in glob.glob(f"{C['LOCAL_OUT']}/{sid}*"):
        os.remove(stale)
    cmd = ["spliceai", "-I", bcf, "-O", out, "-R", C["LOCAL_REF"], "-A", C["ANNOTATION"],
           "-D", str(C["DISTANCE"]), "-G", "all", "--precision", C["PRECISION"],
           "-B", str(C["PRED_BATCH"]), "-T", str(C["TORCH_BATCH"]),
           "--batch-workers", str(C["BATCH_WORKERS"]), "-t", C["LOCAL_TMP"], "-V"] + C["EXTRA_FLAGS"].split()
    log(f"{sid}: start ({s['chrom']}:{s['start']}-{s['end']})")
    t0 = time.time()
    env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    rc = subprocess.run(cmd, env=env).returncode
    if rc != 0 or not os.path.exists(out):
        log(f"{sid}: FAILED rc={rc} — stopping"); sys.exit(rc or 1)
    partial = final + ".partial"
    shutil.move(out, partial)      # sequential copy to Drive
    os.replace(partial, final)     # atomic commit
    log(f"{sid}: done in {(time.time()-t0)/3600:.2f} h -> {final}")
    shutil.rmtree(C["LOCAL_TMP"], ignore_errors=True); os.makedirs(C["LOCAL_TMP"], exist_ok=True)
log("ALL SHARDS DONE")


Launch (re-run after any disconnect to resume):

In [ ]:
import subprocess, time
if subprocess.run(["pgrep", "-f", "python /content/[c]olab_driver.py"], capture_output=True).returncode == 0:
    print("driver already running — see the monitor cell")
else:
    os.makedirs(DRIVE_LOGS, exist_ok=True)
    logf = f"{DRIVE_LOGS}/driver_{time.strftime('%Y%m%d_%H%M%S')}.log"
    subprocess.Popen(f"nohup python /content/colab_driver.py > {logf} 2>&1 &", shell=True)
    print("driver launched; log:", logf)


## 8. Monitor

In [ ]:
import glob
logs = sorted(glob.glob(f"{DRIVE_LOGS}/driver_*.log"))
print(open(logs[-1]).read()[-3000:] if logs else "no log yet")
print(f"\n{len(glob.glob(f'{DRIVE_OUT}/shard_*.tsv.gz'))} / {len(json.load(open(MANIFEST)))} shards finished on Drive")
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!pgrep -af "python /content/[c]olab_driver.py" || echo "driver not running"


## 9. Merge (after all shards finish)

Concatenates shard TSVs with a single header. Pure I/O — run on a CPU runtime to save credits.

In [ ]:
%%bash
set -e
eval "$(python -c "import json;c=json.load(open('/content/run_config.json'));print(f'DRIVE_OUT={c[\"DRIVE_OUT\"]};DRIVE_ROOT={c[\"DRIVE_ROOT\"]}')")"
if ls "$DRIVE_OUT"/*.partial >/dev/null 2>&1; then echo "partial files present — a shard is still copying"; exit 1; fi
OUT="$DRIVE_ROOT/spliceai_all.tsv.gz"
FIRST=$(ls "$DRIVE_OUT"/shard_*.tsv.gz | head -1)
{ zcat "$FIRST" | head -1
  for f in "$DRIVE_OUT"/shard_*.tsv.gz; do zcat "$f" | tail -n +2; done; } | pigz > "$OUT"
echo "merged -> $OUT"; zcat "$OUT" | wc -l
